In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

train_url = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt"
raw_df = pd.read_csv(train_url, sep='\t')

raw_df = raw_df.dropna().reset_index(drop=True)



_, sample_df = train_test_split(
    raw_df, #쪼갤 대상이 되는 원본 데이터프레임입니다.
    test_size=10000,
    stratify=raw_df['label'], #층화추출, lable의 비율을 유지시킴
    random_state=42
)
sample_df = sample_df.reset_index(drop=True)



texts = sample_df['document'].tolist()
labels = sample_df['label'].tolist()


# 결과 확인하기
print(f"구성된 텍스트(texts) 개수: {len(texts)}개")
print(f"구성된 라벨(labels) 개수: {len(labels)}개")
print()
print(f"샘플 데이터 text[0] : {texts[0]}")
print(f"샘플 데이터 label[0]: {labels[0]} (0: 부정, 1: 긍정)")
print(f"샘플 데이터 text[1] : {texts[1]}")
print(f"샘플 데이터 label[1]: {labels[1]} (0: 부정, 1: 긍정)")

위 코드는 미션 1 데이터를 가져오고, 정제하고, 리스트로 나누고 확인한 것

In [ ]:
!pip install -q transformers convertdate pydantic==1.10.11

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 'beomi/kcbert-base' 모델을 사용합니다.
model_name = "beomi/kcbert-base"

print(f"[{model_name}] 모델 및 토큰나이저를 불러오는 중입니다.")


tokenizer = AutoTokenizer.from_pretrained(model_name)


model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


print(f"사용된 한국어 모델: {model_name}")
print(f"Tokenizer 로드 완료: {type(tokenizer).__name__}")
print(f"Sequence Classification 모델 로드 완료: {type(model).__name__}")
print(f"모델 출력 라벨 개수(num_labels): {model.config.num_labels} (긍정/부정 2진 분류 전용)")

텍스트를 분석하기 위해 transformers, BERT 모델을 불러옵니다.

In [ ]:
!pip install -q datasets

In [ ]:
from datasets import Dataset


raw_dataset = Dataset.from_dict({
    'text': texts,
    'label': labels
})

print("1. 초기 Dataset 구조 확인:")
print(raw_dataset)


raw_dataset = raw_dataset.class_encode_column("label")



dataset_split = raw_dataset.train_test_split(
    test_size=0.2,
    stratify_by_column='label',
    seed=42
)

train_dataset = dataset_split['train']
val_dataset = dataset_split['test']  



print(f"최종 분할된 데이터셋 구조:\n{dataset_split}")
print()
print(f"학습용(Train) 데이터 개수: {train_dataset.num_rows}개")
print(f"검증용(Validation) 데이터 개수: {val_dataset.num_rows}개")
print()
print(f"Train 데이터 첫 번째 샘플 확인:\n{train_dataset[0]}")

HuggingFace 의 기능을 사용하기 위해서 데이터를 재가공하는 것입니다.

In [ ]:
def tokenize_function(examples):
    """
    허깅페이스 Dataset 내의 'text' 컬럼을 받아 BERT 모델 입력용 숫자로 변환합니다.
    """
    return tokenizer(
        examples["text"],           
        truncation=True,            
        padding="max_length",       
        max_length=128              # 문장의 최대 길이를 128 로 지정합니다.
    )

print("Tokenizing 함수 정의 완료!")
print()


tokenized_datasets = dataset_split.map(tokenize_function, batched=True)


train_tokenized = tokenized_datasets["train"]
val_tokenized = tokenized_datasets["test"]


print(f"전처리(Tokenizing) 완료 후 데이터셋 구조:")
print(tokenized_datasets)
print()
print(f"토큰화된 Train 데이터의 첫 번째 샘플에 추가된 피처 확인:")
print(f" - 생성된 피처 종류: {list(train_tokenized.features.keys())}")
print(f" - input_ids(숫자 변환 결과)의 길이: {len(train_tokenized[0]['input_ids'])} (지정한 max_length=128과 일치!)")
print(f" - attention_mask(패딩 위치 표시)의 길이: {len(train_tokenized[0]['attention_mask'])}")

text데이터를 tokenizing하고, 규격화 합니다.

In [ ]:
!pip install -q evaluate scikit-learn

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",          
    learning_rate=2e-5,              
    per_device_train_batch_size=16,  
    per_device_eval_batch_size=16,   
    num_train_epochs=3,              
    weight_decay=0.01,               
    logging_steps=100,               
    fp16=True,                       


    eval_strategy="no",              
    save_strategy="epoch"            
)



trainer = Trainer(
    model=model,                         
    args=training_args,                  
    train_dataset=train_tokenized,       
    eval_dataset=val_tokenized           
)

print("Fine-Tuning을 시작합니다.")
print()

# 학습 시작 명령!
trainer.train()

print()
print("학습완료")

HuggingFace 기반 Fine-Tuning 방식으로 학습을 진행합니다.

In [ ]:
import numpy as np
import evaluate

print("검증 데이터셋 성능 평가")


metric = evaluate.load("accuracy")



def compute_metrics(eval_pred):
    logits, labels = eval_pred
    
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)



trainer.compute_metrics = compute_metrics

eval_results = trainer.evaluate()


# 최종 결과 출력
print()
print(f"검증 데이터셋 기준 최종 정확도(Accuracy): {eval_results['eval_accuracy'] * 100:.2f}%")
print(f"검증 데이터셋 기준 최종 손실값(Loss): {eval_results['eval_loss']:.4f}")
print()

모델의 성능을 평가합니다.

In [ ]:
import torch

print("실전 문장 감정 분석 테스트")

test_sentences = [
    "보는 내내 도파민 돌았다 ㄷㄷ 확실히 좀비 영화는 영화관에서 봐야 쫄리는 듯",
    "구교환은 ’코리안 조커‘가 맞다",
    "피곤할때보면 중간넘어서 깜빡 잠들수도 있음 꿈꾸듯 몰입감은 좋은데 밋밋함",
    "좀비 이젠 식상하다못해 졸린다;;; 부산행 킹덤 그 이상 나올수가 없어..."
]


model.eval() 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"현재 추론에 사용 중인 연산 디바이스: {device}\n")
print("-" * 60)

for idx, sentence in enumerate(test_sentences, 1):

    
    inputs = tokenizer(
        sentence,
        return_tensors="pt", 
        truncation=True,      
        padding="max_length", 
        max_length=128        
    )

   
    inputs = {k: v.to(device) for k, v in inputs.items()}


    with torch.no_grad():
        outputs = model(**inputs)

   
    logits = outputs.logits
    predicted_class = torch.argmax(logits, dim=-1).item()

    
    sentiment = "긍정" if predicted_class == 1 else "부정"

    print(f"[{idx}] 입력 문장: \"{sentence}\"")
    print(f"        분석 결과: {sentiment}")
    print()

실제 문장을 분석

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
MY_MODEL_NAME = "chozae/nsmc-sentiment" #본인 아이디로 설정

model.push_to_hub(MY_MODEL_NAME)
tokenizer.push_to_hub(MY_MODEL_NAME)

print("업로드 완료!")
print(f"모델 주소: https://huggingface.co/{MY_MODEL_NAME}")

In [ ]:
from huggingface_hub import HfApi

MY_MODEL_NAME = "chozae/nsmc-sentiment" # 위에서 설정한 모델명으로 변경
MY_SPACE_NAME = "chozae/gdg_ai_mission" # 본인이 설정한 spacename으로 변경

# HuggingFace Hub에 업로드한 감정 분석 모델을 불러와 Gradio 웹앱으로 실행하는 코드입니다.
# Gradio UI 코드 부분은 자유롭게 구성하셔도 됩니다.
app_code = f'''
import gradio as gr
from transformers import pipeline

classifier = pipeline("text-classification", model="{MY_MODEL_NAME}")

def format_result(result):
    label = result["label"]
    score = result["score"]

    if label == "LABEL_1":
        emoji, label_kr = "😊", "긍정"
    else:
        emoji, label_kr = "😞", "부정"

    return f"{{emoji}} {{label_kr}} (확신도: {{score:.1%}})"

def predict(text):
    if not text.strip():
        return "문장을 입력해주세요."
    result = classifier(text)[0]
    return format_result(result)

demo = gr.Interface(
    fn=predict,
    inputs=gr.Textbox(label="영화 리뷰", placeholder="리뷰를 입력하세요...", lines=3),
    outputs=gr.Textbox(label="감정 분석 결과"),
    title="AI 영화 리뷰 감정 분석기",
    description="NSMC 데이터로 파인튜닝된 한국어 감정 분석 모델입니다.",
    examples=[
        ["이 영화 진짜 재미있어요!"],
        ["완전 지루하고 별로였음"],
        ["배우 연기는 좋았지만 스토리가 아쉬웠다"]
    ]
)
demo.launch()
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(app_code)

with open('requirements.txt', 'w') as f:
    f.write("transformers\ngradio\ntorch\n")

# Spaces에 업로드
api = HfApi()
api.upload_file(
    path_or_fileobj="app.py",
    path_in_repo="app.py",
    repo_id=MY_SPACE_NAME,
    repo_type="space"
)
api.upload_file(
    path_or_fileobj="requirements.txt",
    path_in_repo="requirements.txt",
    repo_id=MY_SPACE_NAME,
    repo_type="space"
)

print("완료!")
print(f"https://huggingface.co/spaces/{MY_SPACE_NAME}")